In [3]:
import numpy as np
import pandas as pd

data = np.genfromtxt(
    "dataset/train.csv",
    delimiter=",",
    names=True,
    dtype=None,
    encoding="utf-8"
)
df = pd.DataFrame(data)


In [94]:
df

,id,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,flight_class,duration,days_left,price
0,1,Wayfarer Aviation,IX 1070,Metropolis,Evening,0,Night,Springfield,Economy,4.300499,18,8909
1,2,Oceanic Airlines,6E 285,Los Santos,Night,0,Night,Springfield,Economy,1.330000,38,5115
2,3,Wayfarer Aviation,AI 2641,Night City,Evening,0,Night,Springfield,Business,2.080000,13,35247
3,4,Oceanic Airlines,6E 6704,Night City,Afternoon,0,Evening,Metropolis,Economy,2.250000,5,6457
4,5,Oceanic Airlines,6E 234,Hill Valley,Evening,1,Night,Springfield,Economy,4.830000,21,6842
...,...,...,...,...,...,...,...,...,...,...,...,...
15584,15585,None,AI 2471,Night City,Morning,0,Afternoon,Hill Valley,Business,4.300499,39,57425
15585,15586,None,AI 2776,Los Santos,Night,0,Late_Night,Metropolis,Business,2.500000,27,31023
15586,15587,Oceanic Airlines,6E 6515,Metropolis,Night,0,Night,Springfield,Economy,2.920000,19,10727
15587,15588,Oceanic Airlines,6E 449,Metropolis,Early_Morning,0,Early_Morning,Night City,Economy,2.250000,9,6425


### Cleaning up data
- converting all various null values to None (eg, empty string or NaN)
- encoding stops to numerical value
- replace null values in duration by their means

In [95]:
df.airline = df.airline.replace("", None)
df.duration.replace(np.nan, None, inplace=True)

In [96]:
df.stops = df.stops.map({
    "zero": 0,
    "one": 1,
    "two_or_more": 2
})

In [97]:
df.duration.fillna(df.duration.mean(), inplace=True)

In [99]:
df

,id,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,flight_class,duration,days_left,price
0,1,Wayfarer Aviation,IX 1070,Metropolis,Evening,NaN,Night,Springfield,Economy,4.300499,18,8909
1,2,Oceanic Airlines,6E 285,Los Santos,Night,NaN,Night,Springfield,Economy,1.330000,38,5115
2,3,Wayfarer Aviation,AI 2641,Night City,Evening,NaN,Night,Springfield,Business,2.080000,13,35247
3,4,Oceanic Airlines,6E 6704,Night City,Afternoon,NaN,Evening,Metropolis,Economy,2.250000,5,6457
4,5,Oceanic Airlines,6E 234,Hill Valley,Evening,NaN,Night,Springfield,Economy,4.830000,21,6842
...,...,...,...,...,...,...,...,...,...,...,...,...
15584,15585,None,AI 2471,Night City,Morning,NaN,Afternoon,Hill Valley,Business,4.300499,39,57425
15585,15586,None,AI 2776,Los Santos,Night,NaN,Late_Night,Metropolis,Business,2.500000,27,31023
15586,15587,Oceanic Airlines,6E 6515,Metropolis,Night,NaN,Night,Springfield,Economy,2.920000,19,10727
15587,15588,Oceanic Airlines,6E 449,Metropolis,Early_Morning,NaN,Early_Morning,Night City,Economy,2.250000,9,6425


In [114]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OrdinalEncoder,
    PolynomialFeatures,
    StandardScaler,
)

train_df = pd.read_csv("dataset/train.csv")
test_df = pd.read_csv("dataset/test.csv")

target = "price"

X = train_df.drop(columns=[target, "id"])
y = train_df[target].astype(float)

test_ids = test_df["id"]
X_test = test_df.drop(columns=["id"])

categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_columns = X.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "label_encoder",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        ),
    ),
    ("normalizer", StandardScaler()),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("normalizer", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_columns),
    ("numeric", numeric_pipeline, numeric_columns),
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("regressor", LinearRegression()),
])

model.fit(X_train, y_train)

validation_predictions = model.predict(X_valid)

mae = mean_absolute_error(y_valid, validation_predictions)
mape = mean_absolute_percentage_error(y_valid, validation_predictions)
rmse = np.sqrt(mean_squared_error(y_valid, validation_predictions))
r2 = r2_score(y_valid, validation_predictions)

print("Categorical columns:", categorical_columns)
print("Numeric columns:", numeric_columns)
print(
    "Processed feature shape:",
    model.named_steps["preprocessor"].transform(X_train).shape,
)
print(f"MAE: {mae:.2f}")
print(f"MAPE: {mape * 100:.2f}%")
print(f"Approximate accuracy: {(1 - mape) * 100:.2f}%")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f} ({r2 * 100:.2f}%)")

print("\nSample validation predictions:")
print(
    pd.DataFrame({
        "actual_price": y_valid.iloc[:5].to_numpy(),
        "predicted_price": np.round(validation_predictions[:5], 2),
    }).to_string(index=False)
)

# Train on all available training data and predict the test data.
model.fit(X, y)

test_predictions = model.predict(X_test)

submission = pd.DataFrame({
    "id": test_ids,
    "price": np.round(test_predictions, 2),
})

submission.to_csv("submission.csv", index=False)

print("\nTest predictions:")
print(submission.head().to_string(index=False))
print(f"\nSaved {len(submission)} predictions to submission.csv")

Categorical columns: ['airline', 'flight', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class']
Numeric columns: ['duration', 'days_left']
Processed feature shape: (12471, 10)
MAE: 9746.31
MAPE: 47.90%
Approximate accuracy: 52.10%
RMSE: 17896.41
R²: 0.7727 (77.27%)

Sample validation predictions:
 actual_price  predicted_price
     147027.0        119368.83
       7432.0         12663.54
       7459.0          6107.31
       8026.0          4642.24
     130983.0        132530.89

Test predictions:
 id    price
  1  2964.05
  2  9006.64
  3 42016.88
  4 34080.20
  5  5457.93

Saved 3898 predictions to submission.csv


In [112]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['airline','flight','source_city',...,'flight_class','duration', 'days_left']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. 

In [115]:
from sklearn.model_selection import KFold, cross_val_score

# Select the degree using 5-fold cross-validation. Each fold fits its own
# preprocessing steps, preventing validation information from leaking.
degrees = [1, 2, 3]
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for degree in degrees:
    degree_model = Pipeline([
        ("preprocessor", preprocessor),
        (
            "polynomial_features",
            PolynomialFeatures(degree=degree, include_bias=False),
        ),
        ("regressor", LinearRegression()),
    ])

    negative_rmse_scores = cross_val_score(
        degree_model,
        X,
        y,
        cv=kfold,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    rmse_scores = -negative_rmse_scores
    cv_results.append({
        "degree": degree,
        "mean_rmse": rmse_scores.mean(),
        "std_rmse": rmse_scores.std(),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("mean_rmse")
print("Cross-validation results:")
print(cv_results_df.to_string(index=False, formatters={
    "mean_rmse": "{:,.2f}".format,
    "std_rmse": "{:,.2f}".format,
}))

best_degree = int(cv_results_df.iloc[0]["degree"])
print(f"\nSelected degree: {best_degree}")
print(f"Lowest mean CV RMSE: {cv_results_df.iloc[0]['mean_rmse']:,.2f}")

best_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "polynomial_features",
        PolynomialFeatures(degree=best_degree, include_bias=False),
    ),
    ("regressor", LinearRegression()),
])
best_model.fit(X, y)

best_test_predictions = best_model.predict(X_test)
best_submission = pd.DataFrame({
    "id": test_ids,
    "price": np.round(best_test_predictions, 2),
})
best_submission.to_csv("submission_polynomial_cv.csv", index=False)

print("\nSelected-model test predictions:")
print(best_submission.head().to_string(index=False))
print(
    f"\nSaved {len(best_submission)} predictions "
    "to submission_polynomial_cv.csv"
)

Cross-validation results:
 degree mean_rmse std_rmse
      3 16,363.01   822.37
      2 18,459.74   693.64
      1 23,648.33 1,108.24

Selected degree: 3
Lowest mean CV RMSE: 16,363.01

Selected-model test predictions:
 id    price
  1   227.44
  2  9310.88
  3 32987.60
  4 37915.02
  5 12003.72

Saved 3898 predictions to submission_polynomial_cv.csv
